# Edinburgh housing market — exploration

Run the monitor first so there is data to look at:

```
.venv\Scripts\python -m rightmove_monitor.cli all
```

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
P = ROOT / 'data' / 'processed'
pd.options.display.float_format = lambda v: f'{v:,.0f}'

## UK HPI — sold-price benchmark and forecast

In [ ]:
fc = pd.read_csv(P / 'forecast_ukhpi.csv', parse_dates=['date'])
act = fc[fc.kind == 'actual']
fut = fc[fc.kind == 'forecast']

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(act.date.tail(120), act.avg_price.tail(120), label='UK HPI average price')
ax.plot(fut.date, fut.avg_price, '--', label='forecast')
ax.fill_between(fut.date, fut.lo95, fut.hi95, alpha=.15, label='95% interval')
ax.set_title('City of Edinburgh average house price'); ax.legend(); ax.grid(alpha=.3)
plt.show()

## Rightmove asking prices and inventory over time

Meaningful once you have several weeks of daily snapshots.

In [ ]:
ts = pd.read_csv(P / 'market_timeseries.csv', parse_dates=['snapshot_date'])
display(ts.tail())

if len(ts) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(ts.snapshot_date, ts.median_price); axes[0].set_title('Median asking price (£)')
    axes[1].plot(ts.snapshot_date, ts.n_listings); axes[1].set_title('Live listings')
    for a in axes: a.grid(alpha=.3)
    plt.show()

In [ ]:
seg = pd.read_csv(P / 'market_timeseries_by_segment.csv', parse_dates=['snapshot_date'])
latest = seg[seg.snapshot_date == seg.snapshot_date.max()]
latest[latest.segment_kind == 'outcode'].sort_values('median_price', ascending=False)

## Listing-level history — discounting and time on market

In [ ]:
hist = pd.read_parquet(P / 'listings_history.parquet')
print(f"{len(hist):,} listings tracked, {(hist.status=='active').sum():,} active")

reduced = hist[hist.n_price_changes > 0].copy()
reduced['pct_cut'] = 100 * (reduced.last_price / reduced.first_price - 1)
reduced[['outcode', 'bedrooms', 'first_price', 'last_price', 'pct_cut', 'days_on_market_observed']].sort_values('pct_cut').head(15)